In [1]:
# Installation commands
!pip install SpeechRecognition
!pip install pydub
!pip install gTTS

import speech_recognition as sr
from difflib import get_close_matches
from gtts import gTTS
from IPython.display import Audio, display
import io
import re
import csv
from google.colab import files
import pandas as pd

In [2]:
# ===== DEFAULT TELEPHONE DIRECTORY =====
DIRECTORY = {
    "ANITA": {"phone": "9876543210", "address": "123 Main St, Mumbai"},
    "ANITHA": {"phone": "9876543211", "address": "456 Park Ave, Delhi"},
    "JOHN": {"phone": "9123456780", "address": "789 Oak Rd, Bangalore"},
    "DAVID": {"phone": "9234567890", "address": "321 Pine St, Chennai"},
    "SARAH": {"phone": "9345678901", "address": "654 Elm Dr, Pune"},
    "MICHAEL": {"phone": "9456789012", "address": "987 Maple Ln, Hyderabad"},
    "EMMA": {"phone": "9567890123", "address": "147 Cedar Ct, Kolkata"},
    "ROBERT": {"phone": "9678901234", "address": "258 Birch Way, Ahmedabad"},
    "LISA": {"phone": "9789012345", "address": "369 Willow St, Jaipur"},
    "JAMES": {"phone": "9890123456", "address": "741 Spruce Ave, Lucknow"}
}

In [3]:
def upload_csv_file():
    """
    Upload CSV file in Google Colab
    Returns the uploaded filename
    """
    print("\n Please upload your CSV file...")
    print("\n")

    uploaded = files.upload()

    if uploaded:
        filename = list(uploaded.keys())[0]
        print(f" File uploaded: {filename}")
        return filename
    else:
        print(" No file uploaded")
        return None

In [4]:
def load_directory_from_csv(csv_path=None):
    """
    Load directory from CSV or Excel file
    Supported formats: .csv, .xlsx, .xls
    Handles multiple encodings automatically
    """
    directory = {}

    if not csv_path:
        print(" No file path provided")
        return DIRECTORY

    # Detect file type
    file_extension = csv_path.lower().split('.')[-1]

    df = None
    used_encoding = None

    try:
        # Handle Excel files
        if file_extension in ['xlsx', 'xls']:
            print(f"Reading Excel file: {csv_path}")
            try:
                df = pd.read_excel(csv_path, engine='openpyxl' if file_extension == 'xlsx' else None)
                print(f" Successfully read Excel file")
            except Exception as e:
                print(f" Error reading Excel file: {e}")
                print("\nTrying alternative method...")
                try:
                    df = pd.read_excel(csv_path)
                    print(f" Successfully read Excel file with alternative method")
                except Exception as e2:
                    print(f"Failed: {e2}")
                    return DIRECTORY

        # Handle CSV files
        elif file_extension == 'csv':
            print(f" Reading CSV file: {csv_path}")
            # List of encodings to try
            encodings = ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252', 'windows-1252', 'utf-16']

            for encoding in encodings:
                try:
                    df = pd.read_csv(csv_path, encoding=encoding)
                    used_encoding = encoding
                    print(f" Successfully read CSV with {encoding} encoding")
                    break
                except (UnicodeDecodeError, UnicodeError):
                    continue
                except Exception as e:
                    continue

            if df is None:
                print(" Could not read CSV with any standard encoding")
                print("Please save your CSV as UTF-8 encoding and try again")
                return DIRECTORY

        else:
            print(f" Unsupported file format: .{file_extension}")
            print("Supported formats: .csv, .xlsx, .xls")
            return DIRECTORY

        # Display first few rows for verification
        print(f"\n Preview of loaded data:")
        print(df.head(3).to_string(index=False))
        print(f"\nTotal rows in CSV: {len(df)}")

        # Check for required columns
        required_cols = ['Name', 'Phone', 'Address']

        # Handle case-insensitive column names
        df.columns = df.columns.str.strip()
        col_mapping = {col.lower(): col for col in df.columns}

        # Try to find columns (case-insensitive)
        name_col = None
        phone_col = None
        address_col = None

        for col in df.columns:
            col_lower = col.lower()
            if 'name' in col_lower:
                name_col = col
            elif 'phone' in col_lower or 'number' in col_lower:
                phone_col = col
            elif 'address' in col_lower:
                address_col = col

        if not all([name_col, phone_col, address_col]):
            print(f" Warning: Could not find all required columns")
            print(f"Found columns: {list(df.columns)}")
            print("Using default directory instead")
            return DIRECTORY

        # Load data into dictionary
        loaded_count = 0
        skipped_count = 0

        for idx, row in df.iterrows():
            try:
                name = str(row[name_col]).strip().upper()
                phone = str(row[phone_col]).strip()
                address = str(row[address_col]).strip()

                # Skip empty or invalid rows
                if not name or name == 'NAN' or not phone or phone == 'NAN':
                    skipped_count += 1
                    continue

                directory[name] = {
                    "phone": phone,
                    "address": address if address != 'nan' else "Address not available"
                }
                loaded_count += 1

            except Exception as e:
                print(f" Skipping row {idx + 1}: {e}")
                skipped_count += 1
                continue

        print(f"Successfully loaded {loaded_count} entries from CSV")
        if skipped_count > 0:
            print(f" Skipped {skipped_count} invalid/empty rows")
        print(f"Sample entries: {', '.join(list(directory.keys())[:5])}")

        if len(directory) == 0:
            print(" No valid entries found in CSV. Using default directory.")
            return DIRECTORY

        return directory

    except FileNotFoundError:
        print(f" CSV file not found: {csv_path}")
        return DIRECTORY
    except Exception as e:
        print(f" Error loading CSV: {e}")
        print("Using default directory instead")
        return DIRECTORY


In [5]:
def recognize_speech_from_microphone(duration=10):
    recognizer = sr.Recognizer()

    print(f" Listening for {duration} seconds...")
    print("Please spell the name letter by letter (e.g., A N I T A)")

    try:
        # For Google Colab, use this approach
        from google.colab import output
        from base64 import b64decode
        from io import BytesIO

        # JavaScript to capture audio
        RECORD = """
        const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
        const b2text = blob => new Promise(resolve => {
          const reader = new FileReader()
          reader.onloadend = e => resolve(e.srcElement.result)
          reader.readAsDataURL(blob)
        })

        var record = time => new Promise(async resolve => {
          stream = await navigator.mediaDevices.getUserMedia({ audio: true })
          recorder = new MediaRecorder(stream)
          chunks = []
          recorder.ondataavailable = e => chunks.push(e.data)
          recorder.start()
          await sleep(time)
          recorder.onstop = async ()=>{
            blob = new Blob(chunks)
            text = await b2text(blob)
            resolve(text)
          }
          recorder.stop()
        })
        """

        from IPython.display import Javascript
        display(Javascript(RECORD))

        # Record audio
        s = output.eval_js(f'record({duration * 1000})')

        # Convert to audio data
        b = b64decode(s.split(',')[1])

        # Process with speech recognition
        with sr.AudioFile(BytesIO(b)) as source:
            audio = recognizer.record(source)
            text = recognizer.recognize_google(audio)
            return text

    except Exception as e:
        print(f" Error during speech recognition: {e}")
        print("\nFallback: Please type the letters manually")
        return None


In [6]:
def recognize_from_audio_file(audio_path):
    """
    Recognize speech from uploaded audio file
    """
    recognizer = sr.Recognizer()

    try:
        with sr.AudioFile(audio_path) as source:
            audio = recognizer.record(source)
            text = recognizer.recognize_google(audio)
            return text
    except Exception as e:
        print(f" Error: {e}")
        return None

In [7]:
def parse_spelled_name(recognized_text):
    """
    Extract individual letters from recognized speech
    Handles formats like "A N I T A" or "A-N-I-T-A" or "A. N. I. T. A."
    """
    if not recognized_text:
        return []

    # Remove common words and clean text
    text = recognized_text.upper()

    # Extract single letters (A-Z)
    letters = re.findall(r'\b[A-Z]\b', text)

    return letters

In [8]:
def reconstruct_name(letters):
    return ''.join(letters)

In [9]:
def lookup_directory(name, directory, fuzzy=True):
    """
    Search name in directory with optional fuzzy matching
    Handles names with/without spaces intelligently
    Returns: (phone, address, suggestions)
    """
    name = name.upper()

    # Exact match
    if name in directory:
        return directory[name]['phone'], directory[name]['address'], None

    # Try removing all spaces for comparison
    name_no_space = name.replace(" ", "")

    for key in directory.keys():
        key_no_space = key.replace(" ", "")
        if key_no_space == name_no_space:
            print(f" Matched '{name}' to '{key}' (ignoring spaces)")
            return directory[key]['phone'], directory[key]['address'], None

    # Try adding single space between words (for names like BANKSJIM -> BANKS JIM)
    if " " not in name and len(name) > 3:
        # Try splitting at different positions
        for i in range(3, len(name) - 2):
            name_with_space = name[:i] + " " + name[i:]
            if name_with_space in directory:
                print(f" Matched '{name}' to '{name_with_space}'")
                return directory[name_with_space]['phone'], directory[name_with_space]['address'], None

            # Try with double space (common in your data)
            name_with_double_space = name[:i] + "  " + name[i:]
            if name_with_double_space in directory:
                print(f" Matched '{name}' to '{name_with_double_space}'")
                return directory[name_with_double_space]['phone'], directory[name_with_double_space]['address'], None

    # Fuzzy match (also compare without spaces)
    if fuzzy:
        # First try fuzzy on original names
        matches = get_close_matches(name, directory.keys(), n=3, cutoff=0.6)

        # If no match, try fuzzy without spaces
        if not matches:
            directory_no_spaces = {k.replace(" ", ""): k for k in directory.keys()}
            matches_no_space = get_close_matches(name_no_space, directory_no_spaces.keys(), n=3, cutoff=0.6)
            if matches_no_space:
                matches = [directory_no_spaces[m] for m in matches_no_space]

        if matches:
            return None, None, matches

    return None, None, None

In [10]:
def text_to_speech(text):
    """
    Convert text to speech using gTTS
    """
    try:
        tts = gTTS(text=text, lang='en', slow=False)
        fp = io.BytesIO()
        tts.write_to_fp(fp)
        fp.seek(0)
        return Audio(fp.read(), autoplay=True)
    except Exception as e:
        print(f"TTS Error: {e}")
        return None

In [11]:
def format_output(letters, name, phone, address, suggestions=None):
    """
    Format and display results
    """
    print("\n" + "="*50)
    print("===== SPEECH RECOGNITION OUTPUT =====")
    print("="*50)
    print(f"Recognized Letters: {' '.join(letters)}")

    print("\n" + "="*50)
    print("===== NAME RECONSTRUCTION =====")
    print("="*50)
    print(f"Final Name: {name}")

    print("\n" + "="*50)
    print("===== DIRECTORY LOOKUP RESULT =====")
    print("="*50)

    if phone and phone != "Name not found":
        print(f" FOUND!")
        print(f"Phone Number: {phone}")
        print(f"Address: {address}")
        # Format phone number for better speech
        phone_spoken = ' '.join(phone)
        result_text = f"Contact found. {name}. Phone number is {phone_spoken}. Address is {address}."
    else:
        print(" NOT FOUND")
        print("Phone Number: Name not found")
        result_text = f"Sorry, {name} not found in directory."

        if suggestions:
            print(f"\nDid you mean: {', '.join(suggestions)}?")
            result_text += f" Did you mean {suggestions[0]}?"

    print("="*50)

    return result_text


In [12]:
def main(directory):
    print(" SPEECH-BASED TELEPHONE DIRECTORY LOOKUP")
    print("="*50)
    print("\nDirectory contains:", len(directory), "entries")
    print("Sample names:", ', '.join(list(directory.keys())[:5]))
    print("\n" + "="*50)

    # Choose input method
    print("\nSelect input method:")
    print("1. Microphone (Google Colab)")
    print("2. Type manually")
    print("3. Audio file")

    choice = input("\nEnter choice (1/2/3): ").strip()

    recognized_text = None

    if choice == "1":
        recognized_text = recognize_speech_from_microphone(duration=10)
    elif choice == "2":
        recognized_text = input("Type the letters separated by spaces (e.g., A N I T A): ")
    elif choice == "3":
        audio_path = input("Enter audio file path: ")
        recognized_text = recognize_from_audio_file(audio_path)

    if not recognized_text:
        print("\n No input received. Using manual input.")
        recognized_text = input("Type the letters separated by spaces: ")

    # Process the input
    letters = parse_spelled_name(recognized_text)

    if not letters:
        print(" No letters recognized!")
        return

    # Reconstruct name
    name = reconstruct_name(letters)

    # Lookup in directory
    phone, address, suggestions = lookup_directory(name, directory)

    # Format and display output
    result_text = format_output(letters, name, phone, address, suggestions)

    # Optional: Text-to-Speech
    use_tts = input("\n Play result as audio? (y/n): ").strip().lower()
    if use_tts == 'y':
        audio = text_to_speech(result_text)
        if audio:
            display(audio)


In [13]:
def quick_lookup(spelled_name, directory=None):
    """
    Quick lookup function for testing
    Usage: quick_lookup("A N I T A")
    """
    if directory is None:
        directory = DIRECTORY

    letters = parse_spelled_name(spelled_name)
    name = reconstruct_name(letters)
    phone, address, suggestions = lookup_directory(name, directory)
    format_output(letters, name, phone, address, suggestions)

In [14]:
if __name__ == "__main__":
    print("\n Starting Telephone Directory Lookup System...")


    # Ask user if they want to upload CSV
    print("Choose directory source:")
    print("1. Use default directory (10 sample entries)")
    print("2. Upload CSV file")

    source_choice = input("\nEnter choice (1/2): ").strip()

    current_directory = DIRECTORY

    if source_choice == "2":
        csv_file = upload_csv_file()
        if csv_file:
            current_directory = load_directory_from_csv(csv_file)
        else:
            print("Using default directory")

    main(current_directory)

    # Allow multiple lookups
    while True:
        continue_search = input("\n\nPerform another lookup? (y/n): ").strip().lower()
        if continue_search != 'y':
            print("\n Thank you for using the directory lookup system!")
            break
        print("\n" + "="*50 + "\n")
        main(current_directory)


 Starting Telephone Directory Lookup System...
Choose directory source:
1. Use default directory (10 sample entries)
2. Upload CSV file

Enter choice (1/2): 2

 Please upload your CSV file...




Saving PhoneBookDataset.xlsx to PhoneBookDataset (5).xlsx
 File uploaded: PhoneBookDataset (5).xlsx
Reading Excel file: PhoneBookDataset (5).xlsx
 Successfully read Excel file

 Preview of loaded data:
            Name        Phone                               Address
  Abraham  Ralph 202-225-8490       417 CHOB,Louisiana 5th District
     Adams  Alma 202-225-1510 222 CHOB,North Carolina 12th District
Aderholt  Robert 202-225-4876         235 CHOB,Alabama 4th District

Total rows in CSV: 441


/usr/local/lib/python3.12/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


Successfully loaded 441 entries from CSV
Sample entries: ABRAHAM  RALPH, ADAMS  ALMA, ADERHOLT  ROBERT, AGUILAR  PETE, ALLEN  RICK
 SPEECH-BASED TELEPHONE DIRECTORY LOOKUP

Directory contains: 441 entries
Sample names: ABRAHAM  RALPH, ADAMS  ALMA, ADERHOLT  ROBERT, AGUILAR  PETE, ALLEN  RICK


Select input method:
1. Microphone (Google Colab)
2. Type manually
3. Audio file

Enter choice (1/2/3): 2
Type the letters separated by spaces (e.g., A N I T A): B A N K S J I M
 Matched 'BANKSJIM' to 'BANKS  JIM' (ignoring spaces)

===== SPEECH RECOGNITION OUTPUT =====
Recognized Letters: B A N K S J I M

===== NAME RECONSTRUCTION =====
Final Name: BANKSJIM

===== DIRECTORY LOOKUP RESULT =====
 FOUND!
Phone Number: 202-225-4436
Address: 509 CHOB,Indiana 3rd District

 Play result as audio? (y/n): y




Perform another lookup? (y/n): n

 Thank you for using the directory lookup system!
